# Ingest a weekly science calendar

Each week the short-term scheduler delivers a science calendar (`<calendar_id>-R<revision>.xml`). Ingesting one:

- records every `Observation_Sequence` as an observation under `data/calendars/`, status `REQUESTED`,
- keeps the full `<Meta>` element verbatim (`Calendar_Status` is recorded but never blocks ingest),
- cross-checks the scheduler's claimed totals against what the file actually contains,
- marks earlier revisions of the same calendar superseded.

Re-ingesting an unchanged file is a no-op, so running this notebook twice is harmless.

In [1]:
from pathlib import Path

from pandoraobservations.calendars import ingest_calendar
from pandoraobservations.database import ObservationDatabase, init_data_dir

repo = Path.cwd().parent
data_dir = init_data_dir(repo / "data")  # idempotent; the marker file makes data/ discoverable
calendar_xml = repo / "examples" / "PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml"

record_path = ingest_calendar(calendar_xml, data_dir=data_dir)
record_path

2026-08-21 16:22:20 WARNING: PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml claims 26 visits / 227       
sequences but contains 25 / 226; recording both.

WindowsPath('N:/Joe Documents/Gits/pandora-observations/data/calendars/PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.json')

The record header keeps both what the scheduler claimed and what the file contained. A disagreement (like this week's) is recorded and warned about, never raised.

In [2]:
db = ObservationDatabase(data_dir)
record = db.read_record("calendars", record_path.name)
header = record["calendar"]
{key: header[key] for key in (
    "calendar_id", "revision", "calendar_status", "scheduler_version",
    "claimed_visits", "parsed_visits", "claimed_sequences", "parsed_sequences",
)}

{'calendar_id': 'PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831',
 'revision': 2,
 'calendar_status': 'INVALID',
 'scheduler_version': '1.3.0',
 'claimed_visits': 26,
 'parsed_visits': 25,
 'claimed_sequences': 227,
 'parsed_sequences': 226}

In [3]:
# The cache flattens every observation (payload parameters included) into one queryable table.
from pandoraobservations.cache import load_observations

observations = load_observations(data_dir)
print(f"{len(observations)} observations of {observations['target_key'].nunique()} targets")
observations[["obs_id", "target", "priority", "start_utc", "duration_s", "status"]].head(8)

226 observations of 30 targets


,obs_id,target,priority,start_utc,duration_s,status
0,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,G4476152832143994112,0,2026-08-24 00:14:00,960.0,REQUESTED
1,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,1,2026-08-24 00:59:00,3180.0,REQUESTED
2,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,G4476152832143994112,0,2026-08-24 01:53:00,840.0,REQUESTED
3,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,2,2026-08-24 02:36:00,3180.0,REQUESTED
4,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,G4476152832143994112,0,2026-08-24 03:30:00,840.0,REQUESTED
5,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,2,2026-08-24 04:13:00,3180.0,REQUESTED
6,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,G4476152832143994112,0,2026-08-24 05:07:00,780.0,REQUESTED
7,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,2,2026-08-24 05:49:00,3240.0,REQUESTED
